# RAG Persistent — SQLite Pipeline

This notebook demonstrates a Retrieval-Augmented Generation (RAG) pipeline using a **persistent** search index backed by [sqlitesearch](https://github.com/alexeygrigorev/sqlitesearch) (SQLite FTS5).

Pipeline steps:
1. **Load** FAQ documents from the LLM Zoomcamp dataset via HTTP (`FaqHttpLoader`)
2. **Index** the documents into a SQLite database on disk (`SqliteIndex`)
3. **Query** the pipeline with a natural-language question (`RAGBase`)
4. **Answer** is generated by the OpenAI API (`OpenAIClient`)

> **Note:** Requires `OPENAI_API_KEY` to be set in your environment or a `.env` file in the project root.

In [ ]:
import sys
sys.path.insert(0, '..')

from src import FaqHttpLoader, MinsearchIndex, SqliteIndex, RAGBase, OpenAIClient, OllamaClient

In [ ]:
loader = FaqHttpLoader()
docs = loader.load()
print(f"Loaded {len(docs)} documents")

In [ ]:
index = SqliteIndex(docs=docs, db_path="faq_index.db")

In [ ]:
# rag = RAGBase(index=index, llm=OpenAIClient(), course_filter="data-engineering-zoomcamp")
rag = RAGBase(
    index=index,
    llm=OllamaClient(),
    model="granite4.1:3b",
    course_filter="data-engineering-zoomcamp"
    )

In [ ]:
answer = rag.rag("How do I join the course?")
print(answer)

## Persistence Note

The SQLite index is saved to `faq_index.db` on the first run.

On subsequent runs, `SqliteIndex` detects the existing file and loads the index directly from disk — **no HTTP fetch or re-ingestion required**. This makes startup significantly faster once the index has been built.

To force a rebuild, simply delete `faq_index.db` before running the notebook again.